# Collecting and visualizing data

Based on search_data. But plotting the output.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.optimize import curve_fit
# Add current directory to path so we can import local modules
sys.path.append(os.getcwd())

import search_data
import run_evaluation
from load_input_yaml import load_params
from calculator import cornell_potential_ansatz

# Define data path
DATA_DIR = os.path.abspath(os.path.join(os.getcwd(), "../data/20260205"))

def load_all_runs(root_dir):
    runs = []
    print(f"Searching for runs in {root_dir}...")
    match_paths = search_data.find_matching_runs(root_dir, {})
    print(f"Found {len(match_paths)} runs.")
    
    for path in match_paths:
        try:
            yaml_path = os.path.join(path, "input.yaml")
            metro, gauge = load_params(yaml_path)
            
            # Get results
            calc_res = run_evaluation.get_or_calculate(path)
            
            if "error" in calc_res or "plot_meta" not in calc_res:
                continue
                
            runs.append({
                "path": path,
                "metro": metro,
                "gauge": gauge,
                "meta": calc_res["plot_meta"],
                "results": calc_res
            })
        except Exception as e:
            print(f"Error loading {path}: {e}")
            continue
            
    return runs

# Load data
all_runs = load_all_runs(DATA_DIR)
print(f"Loaded {len(all_runs)} valid runs.")


In [ ]:
def get_potential_data(runs, t_min=None, r_min=None, target_force=1.65, **criteria):
    """
    Extracts V(R) data and fit curves for runs matching criteria.
    Can recalculate V(R) (if t_min provided) and r0 (if r_min provided).
    
    Args:
        runs: List of run dictionaries from load_all_runs
        t_min: Minimum time for V(R) extraction (optional)
        r_min: Minimum R for r0 fit (optional)
        target_force: Target force for r0 (default 1.65)
        **criteria: Key-value pairs to filter runs (e.g. epsilon1=0.1, L=16)
        
    Returns:
        List of dicts containing beta, r0, potential data, and fit curves.
    """
    extracted = []
    
    for run in runs:
        # 1. Filter Runs
        match = True
        metro = run["metro"]
        gauge = run["gauge"]
        
        for key, val in criteria.items():
            check_key = "L1" if key == "L" and not hasattr(metro, "L") else key
            
            if hasattr(metro, check_key):
                if getattr(metro, check_key) != val:
                    match = False
                    break
            elif hasattr(gauge, check_key):
                if getattr(gauge, check_key) != val:
                    match = False
                    break
        
        if not match:
            continue
            
        results = run["results"]
        meta = run.get("meta", {})
        
        # 2. Get/Recalculate V(R)
        v_dict = results.get("V_R", {})
        v_err_dict = results.get("V_R_err", {})
        
        # If t_min is requested and we have W(R,T) data, recalculate V(R)
        w_dict = results.get("W_R_T", {})
        if t_min is not None and w_dict:
            # Reconstruct W(R,T) structure
            w_data = defaultdict(list) # R -> [(T, W)]
            for k, val in w_dict.items():
                try:
                    r_str, t_str = k.split(',')
                    r, t = int(r_str), int(t_str)
                    w_data[r].append((t, val))
                except ValueError: continue
            
            # Recalculate V(R)
            new_vs = {}
            new_verrs = {} # We don't have errors for W here, assume uniform/approx
            
            for r, pts in w_data.items():
                # Filter T >= t_min and W > 0
                valid_pts = [(t, w) for t, w in pts if t >= t_min and w > 0]
                if len(valid_pts) < 2: continue
                
                ts = np.array([p[0] for p in valid_pts])
                ws = np.array([p[1] for p in valid_pts])
                
                try:
                    # simple log-linear fit: ln(W) = ln(C) - V*t
                    # weighting is not possible without W errors, assume unweighted
                    p = np.polyfit(ts, np.log(ws), 1)
                    V = -p[0]
                    new_vs[str(r)] = V
                    new_verrs[str(r)] = 0.0 # Cannot estimate error without bootstrap/W_err
                except: continue
            
            if new_vs:
                v_dict = new_vs
                v_err_dict = new_verrs

        if not v_dict:
            continue
            
        rs = sorted([int(k) for k in v_dict.keys()])
        vs = np.array([v_dict[str(r)] for r in rs])
        verrs = np.array([v_err_dict.get(str(r), 0) for r in rs])
        
        # 3. Get/Recalculate r0
        r0_val = results.get("r0")
        r0_err_val = results.get("r0_err")
        cornell = meta.get("cornell_params")

        # If r_min is specified, we must refit Cornell potential
        # Or if we recalculated V(R), we must refit anyway
        should_refit = (r_min is not None) or (t_min is not None)
        
        if should_refit:
            # Filter R range
            fit_mask = (np.array(rs) >= (r_min if r_min else 3)) # Default r_min=3
            fit_rs = np.array(rs)[fit_mask]
            fit_vs = vs[fit_mask]
            fit_errs = verrs[fit_mask]
            
            if len(fit_rs) >= 3:
                try:
                    # Estimate initial guess
                    sigma_guess = (fit_vs[-1] - fit_vs[0]) / (fit_rs[-1] - fit_rs[0]) if len(fit_rs) > 1 else 0.1
                    B_guess = 0.26
                    A_guess = fit_vs[0] - sigma_guess * fit_rs[0] + (B_guess / fit_rs[0])
                    p0 = [A_guess, max(1e-4, sigma_guess), B_guess]
                    
                    # Use errors if available and non-zero
                    sigma_w = fit_errs if np.any(fit_errs > 0) else None
                    if sigma_w is not None:
                         # Replace zeros with mean non-zero error to avoid inf weights
                         mean_err = np.mean(sigma_w[sigma_w > 0])
                         sigma_w[sigma_w <= 0] = mean_err

                    popt, _ = curve_fit(
                        cornell_potential_ansatz, fit_rs, fit_vs, 
                        p0=p0, sigma=sigma_w, absolute_sigma=(sigma_w is not None), maxfev=5000
                    )
                    
                    A, sig, B = popt
                    cornell = {'A': A, 'sigma': sig, 'B': B}
                    
                    # Calculate r0: r^2 * (sigma + B/r^2) = target  => r^2 = (target - B)/sigma
                    num = target_force - B
                    if num > 0 and sig > 0:
                        r0_val = np.sqrt(num / sig)
                        r0_err_val = 0.0 # Error propagation not implemented here
                    else:
                        r0_val = np.nan
                except:
                    r0_val = np.nan
            else:
                r0_val = np.nan

        if r0_val is None or np.isnan(r0_val):
            continue

        # Generate fit curve
        fit_r_curve = None
        fit_v_curve = None
        if cornell:
            fit_r_curve = np.linspace(min(rs), max(rs), 100)
            fit_v_curve = cornell_potential_ansatz(fit_r_curve, **cornell)
            
        extracted.append({
            "beta": metro.beta,
            "L": metro.L1,
            "r0": r0_val,
            "r0_err": r0_err_val,
            "rs": np.array(rs),
            "vs": vs,
            "verrs": verrs,
            "fit_r": fit_r_curve,
            "fit_v": fit_v_curve,
            "cornell": cornell,
            "path": run["path"]
        })
        
    extracted.sort(key=lambda x: x["beta"])
    return extracted

## V(R) potential and sommer parameter

given a range for beta.
fixed epsilon1 and epsilon2. Fixed L

In [ ]:
# Select parameters
L_fixed = 46
eps1 = 0.0
eps2 = 0.0

# User Input for Analysis
t_min_val = 2  # Minimum time for V(R) fit (default usually 1)
r_min_val = 2  # Minimum R for Cornell fit (default usually 3)

print(f"Analyzing L={L_fixed}, eps=({eps1},{eps2}) with t_min={t_min_val}, r_min={r_min_val}")

# Get data (recalculates V(R) and r0 if parameters differ from defaults)
data_L = get_potential_data(all_runs, L=L_fixed, epsilon1=eps1, epsilon2=eps2, 
                            t_min=t_min_val, r_min=r_min_val)

if not data_L:
    print(f"No data found for L={L_fixed}, epsilon1={eps1}, epsilon2={eps2}")
else:
    print(f"Found {len(data_L)} runs for L={L_fixed}")
    
    # Prepare plots
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. Potential V(R)
    ax = axes[0]
    # Use colormap for betas
    betas = [d["beta"] for d in data_L]
    cmap = plt.get_cmap("viridis")
    norm = plt.Normalize(min(betas), max(betas))
    
    for d in data_L:
        beta = d["beta"]
        color = cmap(norm(beta))
        ax.errorbar(d["rs"], d["vs"], yerr=d["verrs"], fmt='o', 
                    label=f"$\\beta={beta:.2f}$", color=color, markersize=4)
        if d["fit_r"] is not None:
            ax.plot(d["fit_r"], d["fit_v"], '-', color=color, alpha=0.5)
            # Mark r0 on the curve
            r0 = d["r0"]
            v_r0 = cornell_potential_ansatz(r0, **d["cornell"])
            ax.plot(r0, v_r0, 's', color=color, markersize=5)
            
    ax.set_title(f"Static Potential $V(R)$ ($L={L_fixed}, t_{{min}}={t_min_val}$)")
    ax.set_xlabel("$R/a$")
    ax.set_ylabel("$aV(R)$")
    ax.grid(True, alpha=0.3)
    
    # Legend management
    handles, labels = ax.get_legend_handles_labels()
    # Filter duplicate labels if any
    by_label = dict(zip(labels, handles))
    if len(by_label) > 10:
        ax.legend(by_label.values(), by_label.keys(), bbox_to_anchor=(1.02, 1), loc='upper left', fontsize='small', ncol=1)
    else:
        ax.legend(by_label.values(), by_label.keys(), loc='best')
    
    # 2. Sommer parameter r0 vs beta
    ax = axes[1]
    r0s = [d["r0"] for d in data_L]
    r0_errs = [d["r0_err"] for d in data_L]
    
    ax.errorbar(betas, r0s, yerr=r0_errs, fmt='o-', capsize=3, color='tab:blue')
    ax.set_title(f"Sommer parameter $r_0/a$ vs $\\beta$ ($r_{{min}}={r_min_val}$)")
    ax.set_xlabel("$\\beta$")
    ax.set_ylabel("$r_0/a$")
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## V(R) potential and sommer parameter fixed Volume

given a range for beta.
fixed epsilon1 and epsilon2. Fixed Volume -> different L values.

In [ ]:
# Import volume calculation function
import calculator
import importlib
importlib.reload(calculator)
from calculator import calculate_volume_from_r0

# 1. Calculate and Analyze Available Volumes
print("Calculating volumes for all runs...")
runs_with_vol = []
vol_vals = []
beta_vals = []
L_vals = []

r0_phys_val = 0.5 # fm

for run in all_runs:
    try:
        L = run["metro"].L1
        beta = run["metro"].beta
        r0 = run["results"].get("r0")
        
        # Use function from calculator.py
        vol = calculate_volume_from_r0(L, r0, r0_phys=r0_phys_val)
        
        if not np.isnan(vol) and vol > 0:
            run["volume"] = vol
            runs_with_vol.append(run)
            vol_vals.append(vol)
            beta_vals.append(beta)
            L_vals.append(L)
    except Exception:
        continue

print(f"Computed volumes for {len(runs_with_vol)} runs.")

if vol_vals:
    # Plot distribution
    plt.figure(figsize=(14, 5))
    
    # Histogram
    plt.subplot(1, 2, 1)
    plt.hist(vol_vals, bins=30, alpha=0.7, color='steelblue', edgecolor='black')
    plt.xlabel(r"Volume [(fm)$^4$]")
    plt.ylabel("Count")
    plt.title("Distribution of Physical Volumes")
    plt.grid(True, alpha=0.3)
    
    # Volume vs Beta
    plt.subplot(1, 2, 2)
    sc = plt.scatter(beta_vals, vol_vals, c=L_vals, cmap='viridis', s=50, alpha=0.8, edgecolor='k', linewidth=0.5)
    plt.colorbar(sc, label='L')
    plt.xlabel(r"$\beta$")
    plt.ylabel(r"Volume [(fm)$^4$]")
    plt.title(r"Volume vs $\beta$ (colored by L)")
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Volume Range: {min(vol_vals):.4f} - {max(vol_vals):.4f} fm^4")

In [ ]:
# 2. Select Volume Region and Plot
target_volume = 3.5 # Set your desired target volume here (e.g. based on histogram)
volume_tolerance = 0.1 # +/- 10% tolerance

print(f"Selecting runs with Volume = {target_volume} +/- {volume_tolerance*100}%")

selected_runs_vol = []
for run in runs_with_vol:
    v = run["volume"]
    if abs(v - target_volume) / target_volume <= volume_tolerance:
        selected_runs_vol.append(run)

print(f"Found {len(selected_runs_vol)} runs in target volume range.")

if selected_runs_vol:
    # Sort by beta for cleaner plots
    selected_runs_vol.sort(key=lambda x: x["metro"].beta)
    
    # Get potential data (reusing existing function)
    # We pass the selected list directly, no criteria filtering needed
    data_vol = get_potential_data(selected_runs_vol, t_min=2, r_min=2)
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # A. Potential
    ax = axes[0]
    betas_vol = [d["beta"] for d in data_vol]
    if betas_vol:
        cmap = plt.get_cmap("viridis")
        norm = plt.Normalize(min(betas_vol), max(betas_vol))
        
        for d in data_vol:
            beta = d["beta"]
            L = d["L"]
            color = cmap(norm(beta))
            ax.errorbar(d["rs"], d["vs"], yerr=d["verrs"], fmt='o', 
                        label=f"$\\beta={beta:.2f}, L={L}$", color=color, markersize=4)
            if d["fit_r"] is not None:
                ax.plot(d["fit_r"], d["fit_v"], '-', color=color, alpha=0.5)
                # Mark r0
                r0 = d["r0"]
                v_r0 = cornell_potential_ansatz(r0, **d["cornell"])
                ax.plot(r0, v_r0, 's', color=color, markersize=5)
                
        ax.set_title(f"Potential $V(R)$ at Volume $\\approx {target_volume}$ fm$^4$")
        ax.set_xlabel("$R/a$")
        ax.set_ylabel("$aV(R)$")
        ax.grid(True, alpha=0.3)
        
        # Legend
        handles, labels = ax.get_legend_handles_labels()
        if len(handles) > 10:
             ax.legend(handles, labels, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize='small', ncol=1)
        else:
             ax.legend(handles, labels, loc='best')

    # B. r0/a vs Beta
    ax = axes[1]
    if betas_vol:
        r0s_vol = [d["r0"] for d in data_vol]
        r0_errs_vol = [d["r0_err"] for d in data_vol]
        Ls_vol = [d["L"] for d in data_vol]
        
        for i in range(len(betas_vol)):
            color = cmap(norm(betas_vol[i]))
            ax.errorbar(betas_vol[i], r0s_vol[i], yerr=r0_errs_vol[i], fmt='o', 
                        color=color, capsize=3)
            # Annotate L values to show which L corresponds to which beta
            ax.text(betas_vol[i], r0s_vol[i], f" L={Ls_vol[i]}", fontsize=8, 
                    verticalalignment='bottom', horizontalalignment='right')
            
        ax.set_title(f"Sommer parameter $r_0/a$ vs $\\beta$ (Fixed Volume)")
        ax.set_xlabel("$\\beta$")
        ax.set_ylabel("$r_0/a$")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No runs found in the specified volume range. Try adjusting target_volume or tolerance.")